# Vision Transformer（CIFAR-10）

このノートブックでは、PyTorchのViTを最小構成で学習し、CIFAR-10の分類を行います。最後に cat.jpg と lion.jpg を推論します。

In [1]:
# 必要なライブラリ
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
from matplotlib import font_manager
from PIL import Image

# 日本語フォント設定（利用可能なものを自動選択）
def set_japanese_font():
    candidates = [
        "IPAexGothic",
        "IPAGothic",
        "Noto Sans CJK JP",
        "Noto Sans JP",
        "TakaoGothic",
        "Yu Gothic",
        "Hiragino Sans",
    ]
    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in candidates:
        if name in available:
            plt.rcParams["font.family"] = name
            break
    plt.rcParams["axes.unicode_minus"] = False

set_japanese_font()

# 再現性
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# デバイス
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


## 1. データ準備（CIFAR-10）

1.AutoEncoderでダウンロード済みのCIFAR-10を使用します。ViTの入力に合わせて224×224にリサイズします。

In [2]:
data_root = "../1.AutoEncoder/data"

train_tf = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

test_tf = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_dataset = datasets.CIFAR10(root=data_root, train=True, download=False, transform=train_tf)
test_dataset = datasets.CIFAR10(root=data_root, train=False, download=False, transform=test_tf)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

class_names = train_dataset.classes
print("classes:", class_names)

classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## 2. ViTの構築

学習済みViTをロードし、出力層をCIFAR-10用に置き換えます。

In [3]:
model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

# 分類ヘッドを置き換え
in_features = model.heads.head.in_features
model.heads.head = nn.Linear(in_features, 10)

# 4GPUを想定してDataParallelを利用
if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    if gpu_count >= 2:
        model = nn.DataParallel(model, device_ids=list(range(min(4, gpu_count))))

model = model.to(device)
print(model)

DataParallel(
  (module): VisionTransformer(
    (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (encoder): Encoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): Sequential(
        (encoder_layer_0): EncoderBlock(
          (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attention): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (dropout): Dropout(p=0.0, inplace=False)
          (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): MLPBlock(
            (0): Linear(in_features=768, out_features=3072, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.0, inplace=False)
            (3): Linear(in_features=3072, out_features=768, bias=True)
            (4): Dropout(p=0.0, inplace=False)
          )
        )
        (encoder_layer_1): EncoderBlock(
          (ln_1): L

## 3. 学習ループ

最小構成で数エポックだけ学習します。

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"epoch {epoch+1}/{num_epochs} loss: {epoch_loss:.4f}")

epoch 1/2 loss: 0.2035
epoch 2/2 loss: 0.0984


## 4. テスト画像（cat.jpg, lion.jpg）の推論

学習済みモデルで2枚の画像を推論します。

In [ ]:
model.eval()

test_imgs = ["cat.jpg", "lion.jpg"]

for name in test_imgs:
    if not os.path.exists(name):
        raise FileNotFoundError(f"{name} が見つかりません")

for name in test_imgs:
    img = Image.open(name).convert("RGB")
    x = test_tf(img).unsqueeze(0).to(device)

    with torch.no_grad():
        x = torch.nn.functional.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False)
        logits = model(x)
        pred = torch.argmax(logits, dim=1).item()

    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{name} -> {class_names[pred]}")
    plt.show()

AssertionError: Caught AssertionError in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/home/yanai-lab/li-l/.conda/envs/env1/lib/python3.8/site-packages/torch/nn/parallel/parallel_apply.py", line 84, in _worker
    output = module(*input, **kwargs)
  File "/home/yanai-lab/li-l/.conda/envs/env1/lib/python3.8/site-packages/torch/nn/modules/module.py", line 1553, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/home/yanai-lab/li-l/.conda/envs/env1/lib/python3.8/site-packages/torch/nn/modules/module.py", line 1562, in _call_impl
    return forward_call(*args, **kwargs)
  File "/home/yanai-lab/li-l/.conda/envs/env1/lib/python3.8/site-packages/torchvision/models/vision_transformer.py", line 291, in forward
    x = self._process_input(x)
  File "/home/yanai-lab/li-l/.conda/envs/env1/lib/python3.8/site-packages/torchvision/models/vision_transformer.py", line 272, in _process_input
    torch._assert(w == self.image_size, f"Wrong image width